In [ ]:
# Manifold-Constrained Hyper-Connections
# It is a DeepSeek architecture idea that modifies the standard Transformer residual connection. 
# The standard Transformer has one residual stream:
# x = x + F(x) where F(x) is usually attention or the MLP block. 
# This is stable because if F(x) is small early in training, the model can behave almost like identity:
# That identity path is one of the main reasons very deep residual networks train well

# Hyper-connections expand the standard residual stream into multiple parallel streams. This architecture 
# learns how to move information between these streams dynamically. The original paper frames this mechanism 
# as an alternative to traditional residual connections. It allows the network to adjust connection strengths 
# between features at different depths and dynamically rearrange layers.
    
# However, expanding the stream width and diversifying connectivity patterns compromises the identity-mapping 
# property of residual connections. This compromise causes architectural instability and scalability issues. 
# Manifold Hyper-Connections (MHC) fixes these vulnerabilities by projecting the residual connection space 
# onto a manifold, successfully restoring the identity-mapping property. mHC, constrains residual matrices to 
# doubly stochastic matrices using Sinkhorn-Knopp projection (A Sinkhorn-Knopp projection is a mathematical operation 
# used to take any standard, unconstrained matrix and force it into becoming a doubly stochastic matrix—a square matrix 
# with non-negative real numbers where every single row and column sums up to exactly 1)

#A manifold is a geometric space that looks flat and simple up close, but curves into a more complex shape from a distance.
# In mathematics and data science, it represents a way to understand intricate, high-dimensional spaces by breaking them down 
# into familiar, lower-dimensional structures.

In [17]:
from dataclasses import dataclass
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# Config
# ============================================================

@dataclass
class Config:
    vocab_size: int = 100
    max_seq_len: int = 16
    d_model: int = 32
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.0
    pad_token_id: int = 0

    # mHC config
    n_streams: int = 4
    sinkhorn_iters: int = 20
    residual_identity_bias: float = 5.0

    @property
    def d_head(self):
        assert self.d_model % self.n_heads == 0
        return self.d_model // self.n_heads

In [ ]:
def sinkhorn(
    logits: torch.Tensor,
    num_iters: int = 20,
    eps: float = 1e-12, # Smaller epsilon is safer for precise convergence
) -> torch.Tensor:
    """
    Convert unconstrained logits into an approximately doubly stochastic matrix.
    Numerically stable against explosion/NaN.

    logits: [..., S, S] Supports arbitrary batch dimensions

    returns:
        A: [..., S, S] doubly stochastic matrix
    """
    # 1. Stable exponentiation (subtract max along the last dimension)
    # This prevents overflow to inf without changing the resulting distribution
    max_logits = torch.max(logits, dim=-1, keepdim=True).values
    A = torch.exp(logits - max_logits)

    # 2. Iterative projection
    for _ in range(num_iters):
        # Row normalization (dim=-1 is the last dimension)
        A = A / (A.sum(dim=-1, keepdim=True) + eps)  
        # Column normalization (dim=-2 is the second-to-last dimension)
        A = A / (A.sum(dim=-2, keepdim=True) + eps)  

    return A

In [ ]:
class PositionalEmbeddingLayer(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.token_embeddings = nn.Embedding(
            num_embeddings=config.vocab_size,
            embedding_dim=config.d_model,
            padding_idx=config.pad_token_id,
        )

        self.position_embeddings = nn.Embedding(
            num_embeddings=config.max_seq_len,
            embedding_dim=config.d_model,
        )

        self.dropout = nn.Dropout(config.dropout)

    def forward(self, input_ids):
        """
        input_ids: [B, T]

        returns:
            x: [B, T, D]
        """

        batch_size, seq_len = input_ids.shape

        token_embs = self.token_embeddings(input_ids)

        position_ids = torch.arange(
            seq_len,
            device=input_ids.device,
        )

        pos_embs = self.position_embeddings(position_ids)

        x = token_embs + pos_embs

        return self.dropout(x)

In [ ]:
class mHCConnection(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.n_streams = config.n_streams
        self.sinkhorn_iters = config.sinkhorn_iters

        # H_pre: read/mix streams into one branch input
        self.pre_logits = nn.Parameter(
            torch.zeros(config.n_streams)
        )

        # H_post: write branch output back into streams
        self.post_logits = nn.Parameter(
            torch.zeros(config.n_streams)
        )

        # H_res: stream-to-stream residual mixing matrix
        self.residual_logits = nn.Parameter(
            torch.zeros(config.n_streams, config.n_streams)
        )

        # Bias toward identity at initialization.
        # Identity is already doubly stochastic.
        self.register_buffer(
            "identity_bias",
            torch.eye(config.n_streams) * config.residual_identity_bias,
        )

    def get_weights(self):
        """
        returns:
            pre_weights:  [S]
            post_weights: [S]
            residual_mix: [S, S]
        """

        # Non-negative read/write weights.
        # I normalize them for scale stability in this toy implementation.
        pre_weights = torch.sigmoid(self.pre_logits)
        pre_weights = pre_weights / (pre_weights.sum() + 1e-6)

        post_weights = torch.sigmoid(self.post_logits)
        post_weights = post_weights / (post_weights.sum() + 1e-6)

        # Doubly stochastic residual stream mixing.
        residual_mix = sinkhorn(
            self.residual_logits + self.identity_bias,
            num_iters=self.sinkhorn_iters,
        )

        return pre_weights, post_weights, residual_mix

    def forward(self, X, branch):
        """
        X:
            [B, T, S, D]

        branch:
            function that maps [B, T, D] -> [B, T, D]

        returns:
            X_next: [B, T, S, D]
        """

        pre_weights, post_weights, residual_mix = self.get_weights()

        # ------------------------------------------------------------
        # 1. Read from streams into normal branch input
        # ------------------------------------------------------------

        branch_input = torch.einsum(
            "btsd,s->btd",
            X,
            pre_weights,
        )
        # [B, T, D]

        # ------------------------------------------------------------
        # 2. Apply normal Transformer branch
        # ------------------------------------------------------------

        branch_output = branch(branch_input)
        # [B, T, D]

        # ------------------------------------------------------------
        # 3. Manifold-constrained residual stream mixing
        # ------------------------------------------------------------

        X_residual = torch.einsum(
            "btsd,sr->btrd",
            X,
            residual_mix,
        )
        # [B, T, S, D]

        # ------------------------------------------------------------
        # 4. Write branch output back into streams
        # ------------------------------------------------------------

        X_next = X_residual + (
            branch_output[:, :, None, :]
            * post_weights[None, None, :, None]
        )
        # [B, T, S, D]

        return X_next

In [ ]:
class PositionalEmbeddingLayer(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.token_embeddings = nn.Embedding(
            num_embeddings=config.vocab_size,
            embedding_dim=config.d_model,
            padding_idx=config.pad_token_id,
        )

        self.position_embeddings = nn.Embedding(
            num_embeddings=config.max_seq_len,
            embedding_dim=config.d_model,
        )

        self.dropout = nn.Dropout(config.dropout)

    def forward(self, input_ids):
        """
        input_ids: [B, T]

        returns:
            x: [B, T, D]
        """

        batch_size, seq_len = input_ids.shape

        token_embs = self.token_embeddings(input_ids)

        position_ids = torch.arange(
            seq_len,
            device=input_ids.device,
        )

        pos_embs = self.position_embeddings(position_ids)

        x = token_embs + pos_embs

        return self.dropout(x)

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.d_model = config.d_model
        self.n_heads = config.n_heads
        self.d_head = config.d_head
        self.max_seq_len = config.max_seq_len

        self.q_proj = nn.Linear(config.d_model, config.d_model)
        self.k_proj = nn.Linear(config.d_model, config.d_model)
        self.v_proj = nn.Linear(config.d_model, config.d_model)

        self.o_proj = nn.Linear(config.d_model, config.d_model)

        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        causal_mask = torch.tril(
            torch.ones(
                config.max_seq_len,
                config.max_seq_len,
                dtype=torch.bool,
            )
        )

        causal_mask = causal_mask.view(
            1,
            1,
            config.max_seq_len,
            config.max_seq_len,
        )

        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x, attention_mask=None):
        """
        x: [B, T, D]
        """

        batch_size, seq_len, d_model = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            batch_size,
            seq_len,
            self.n_heads,
            self.d_head,
        ).transpose(1, 2)

        k = k.view(
            batch_size,
            seq_len,
            self.n_heads,
            self.d_head,
        ).transpose(1, 2)

        v = v.view(
            batch_size,
            seq_len,
            self.n_heads,
            self.d_head,
        ).transpose(1, 2)

        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(self.d_head)

        mask = self.causal_mask[:, :, :seq_len, :seq_len]

        if attention_mask is not None:
            key_padding_mask = attention_mask[:, None, None, :].bool()
            mask = mask & key_padding_mask

        scores = scores.masked_fill(
            ~mask,
            torch.finfo(scores.dtype).min,
        )

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        out = attn_weights @ v

        out = out.transpose(1, 2).contiguous()
        out = out.view(batch_size, seq_len, d_model)

        out = self.o_proj(out)
        out = self.resid_dropout(out)

        if attention_mask is not None:
            out = out * attention_mask[:, :, None].to(out.dtype)

        return out

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(config.d_model, 4 * config.d_model),
            nn.GELU(),
            nn.Linear(4 * config.d_model, config.d_model),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class mHCDecoderBlock(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.attn_hc = mHCConnection(config)
        self.mlp_hc = mHCConnection(config)

        self.attn_ln = nn.LayerNorm(config.d_model)
        self.mlp_ln = nn.LayerNorm(config.d_model)

        self.attn = CausalSelfAttention(config)
        self.mlp = FeedForward(config)

    def forward(self, X, attention_mask=None):
        """
        X: [B, T, S, D]
        """

        X = self.attn_hc(
            X,
            branch=lambda x: self.attn(
                self.attn_ln(x),
                attention_mask=attention_mask,
            ),
        )

        X = self.mlp_hc(
            X,
            branch=lambda x: self.mlp(
                self.mlp_ln(x),
            ),
        )

        if attention_mask is not None:
            X = X * attention_mask[:, :, None, None].to(X.dtype)

        return X

In [ ]:
class mHCGPTDecoder(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.config = config

        self.embedding = PositionalEmbeddingLayer(config)

        self.blocks = nn.ModuleList([
            mHCDecoderBlock(config)
            for _ in range(config.n_layers)
        ])

        self.final_ln = nn.LayerNorm(config.d_model)

        self.lm_head = nn.Linear(
            config.d_model,
            config.vocab_size,
            bias=False,
        )

        self.lm_head.weight = self.embedding.token_embeddings.weight

    def expand_streams(self, x):
        """
        x: [B, T, D]

        returns:
            X: [B, T, S, D]
        """

        return x.unsqueeze(2).repeat(1, 1, self.config.n_streams, 1,)

    def reduce_streams(self, X):
        """
        X: [B, T, S, D]

        Public Hyper-Connections implementations use stream reduction
        after the trunk. Here we use sum reduction.
        """

        return X.sum(dim=2)

    def forward(
        self,
        input_ids,
        attention_mask=None,
        labels=None,
        return_debug=False,
    ):
        """
        input_ids:      [B, T]
        attention_mask: [B, T]
        labels:         [B, T]
        """

        batch_size, seq_len = input_ids.shape

        assert seq_len <= self.config.max_seq_len

        x = self.embedding(input_ids)
        # [B, T, D]

        X = self.expand_streams(x)
        # [B, T, S, D]

        for block in self.blocks:
            X = block(
                X,
                attention_mask=attention_mask,
            )

        x = self.reduce_streams(X)
        # [B, T, D]

        x = self.final_ln(x)

        logits = self.lm_head(x)
        # [B, T, vocab_size]

        loss = None

        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = F.cross_entropy(
                shift_logits.view(-1, self.config.vocab_size),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        outputs = {
            "logits": logits,
            "loss": loss,
        }

        if return_debug:
            pre, post, residual_mix = self.blocks[0].attn_hc.get_weights()

            outputs["attn_pre_weights_layer0"] = pre.detach()
            outputs["attn_post_weights_layer0"] = post.detach()
            outputs["attn_residual_matrix_layer0"] = residual_mix.detach()

        return outputs

    def sample_next_token(
        self,
        next_token_logits,
        temperature=1.0,
        top_k=None,
        top_p=None,
    ):
        if temperature == 0:
            return torch.argmax(
                next_token_logits,
                dim=-1,
                keepdim=True,
            )

        next_token_logits = next_token_logits / temperature

        if top_k is not None:
            top_k = min(top_k, next_token_logits.size(-1))

            values, _ = torch.topk(
                next_token_logits,
                k=top_k,
                dim=-1,
            )

            min_values = values[:, -1].unsqueeze(-1)

            next_token_logits = torch.where(
                next_token_logits < min_values,
                torch.full_like(next_token_logits, float("-inf")),
                next_token_logits,
            )

        if top_p is not None:
            assert 0.0 < top_p <= 1.0

            sorted_logits, sorted_indices = torch.sort(
                next_token_logits,
                descending=True,
                dim=-1,
            )

            sorted_probs = torch.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

            sorted_indices_to_remove = cumulative_probs > top_p

            sorted_indices_to_remove[:, 1:] = (
                sorted_indices_to_remove[:, :-1].clone()
            )
            sorted_indices_to_remove[:, 0] = False

            indices_to_remove = torch.zeros_like(
                next_token_logits,
                dtype=torch.bool,
            )

            indices_to_remove.scatter_(
                dim=-1,
                index=sorted_indices,
                src=sorted_indices_to_remove,
            )

            next_token_logits = next_token_logits.masked_fill(
                indices_to_remove,
                float("-inf"),
            )

        probs = torch.softmax(next_token_logits, dim=-1)

        return torch.multinomial(
            probs,
            num_samples=1,
        )

    @torch.no_grad()
    def generate(
        self,
        input_ids,
        max_new_tokens,
        temperature=1.0,
        top_k=None,
        top_p=None,
        eos_token_id=None,
    ):
        self.eval()

        for _ in range(max_new_tokens):
            input_ids_cond = input_ids[:, -self.config.max_seq_len:]

            attention_mask = (
                input_ids_cond != self.config.pad_token_id
            ).long()

            outputs = self(
                input_ids=input_ids_cond,
                attention_mask=attention_mask,
                labels=None,
            )

            next_token_logits = outputs["logits"][:, -1, :]

            next_token = self.sample_next_token(
                next_token_logits,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
            )

            input_ids = torch.cat(
                [input_ids, next_token],
                dim=1,
            )

            if eos_token_id is not None:
                if torch.all(next_token.squeeze(-1) == eos_token_id):
                    break

        return input_ids

In [ ]:
torch.manual_seed(42)

batch_size = 3
seq_len = 8
vocab_size = 100
pad_token_id = 0

lengths = torch.randint(
    low=1,
    high=seq_len + 1,
    size=(batch_size,),
)

input_ids = torch.randint(
    low=1,
    high=vocab_size,
    size=(batch_size, seq_len),
)

attention_mask = (
    torch.arange(seq_len).unsqueeze(0)
    < lengths.unsqueeze(1)
).long()

input_ids = input_ids.masked_fill(
    attention_mask == 0,
    pad_token_id,
)

labels = input_ids.clone()
labels = labels.masked_fill(
    attention_mask == 0,
    -100,
)

config = Config(
    vocab_size=vocab_size,
    max_seq_len=16,
    d_model=32,
    n_heads=4,
    n_layers=2,
    dropout=0.0,
    pad_token_id=pad_token_id,

    n_streams=4,
    sinkhorn_iters=20,
    residual_identity_bias=5.0,
)

model = mHCGPTDecoder(config)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
    return_debug=True,
)

logits = outputs["logits"]
loss = outputs["loss"]

print("lengths:")
print(lengths)

print("\ninput_ids:")
print(input_ids)

print("\nattention_mask:")
print(attention_mask)

print("\nlabels:")
print(labels)

print("\nlogits shape:")
print(logits.shape)

print("\nloss:")
print(loss)

loss.backward()

print("\nGradient exists for mHC residual logits?")
print(model.blocks[0].attn_hc.residual_logits.grad is not None)

print("\nGradient exists for attention q_proj?")
print(model.blocks[0].attn.q_proj.weight.grad is not None)

print("\nGradient exists for MLP first layer?")
print(model.blocks[0].mlp.net[0].weight.grad is not None)

In [ ]:
R = outputs["attn_residual_matrix_layer0"]

print("\nResidual stream mixing matrix R:")
print(R)

print("\nRow sums of R:")
print(R.sum(dim=-1))

print("\nColumn sums of R:")
print(R.sum(dim=-2))

In [6]:
import torch

eps = 1e-6
A = torch.rand(4, 4)
print(A)

for _ in range(20):
    A = A / (A.sum(dim=-1, keepdim=True) + eps)  # row normalize
    A = A / (A.sum(dim=-2, keepdim=True) + eps)  # column normalize
    print(A)
    print('#'*100)
    

tensor([[0.9909, 0.5538, 0.3741, 0.0644],
        [0.6256, 0.8220, 0.8618, 0.5196],
        [0.9120, 0.0072, 0.5964, 0.6215],
        [0.5292, 0.0849, 0.9323, 0.9668]])
tensor([[0.3679, 0.4601, 0.1650, 0.0364],
        [0.1628, 0.4787, 0.2665, 0.2060],
        [0.3142, 0.0056, 0.2441, 0.3262],
        [0.1550, 0.0556, 0.3245, 0.4315]])
####################################################################################################
tensor([[0.3514, 0.4752, 0.1588, 0.0342],
        [0.1437, 0.4569, 0.2370, 0.1790],
        [0.3471, 0.0067, 0.2717, 0.3547],
        [0.1577, 0.0612, 0.3326, 0.4321]])
####################################################################################################
tensor([[0.3445, 0.4734, 0.1551, 0.0332],
        [0.1413, 0.4565, 0.2321, 0.1742],
        [0.3540, 0.0069, 0.2760, 0.3580],
        [0.1603, 0.0632, 0.3367, 0.4346]])
####################################################################################################
tensor([[0.3424, 0.47

In [7]:
sum([0.3415, 0.4722, 0.1535, 0.0328])

1.0

In [9]:
X= torch.rand(2, 4, 8)
X

tensor([[[0.3642, 0.8333, 0.2569, 0.0778, 0.2424, 0.4923, 0.6499, 0.7362],
         [0.1057, 0.9907, 0.5937, 0.5606, 0.2390, 0.4718, 0.4294, 0.8508],
         [0.1053, 0.9353, 0.5285, 0.5943, 0.8888, 0.2838, 0.6447, 0.6535],
         [0.6196, 0.0497, 0.5953, 0.9974, 0.6720, 0.4915, 0.6581, 0.6456]],

        [[0.2139, 0.7623, 0.9011, 0.2775, 0.2771, 0.8305, 0.6048, 0.8672],
         [0.6301, 0.1374, 0.2400, 0.1103, 0.4322, 0.4852, 0.0837, 0.4832],
         [0.3448, 0.6847, 0.8148, 0.2926, 0.5404, 0.1399, 0.0768, 0.0038],
         [0.4260, 0.0336, 0.5221, 0.9801, 0.4706, 0.9882, 0.5860, 0.7098]]])

In [16]:
X.unsqueeze(2).repeat(1, 1, 4, 1)

tensor([[[[0.3642, 0.8333, 0.2569, 0.0778, 0.2424, 0.4923, 0.6499, 0.7362],
          [0.3642, 0.8333, 0.2569, 0.0778, 0.2424, 0.4923, 0.6499, 0.7362],
          [0.3642, 0.8333, 0.2569, 0.0778, 0.2424, 0.4923, 0.6499, 0.7362],
          [0.3642, 0.8333, 0.2569, 0.0778, 0.2424, 0.4923, 0.6499, 0.7362]],

         [[0.1057, 0.9907, 0.5937, 0.5606, 0.2390, 0.4718, 0.4294, 0.8508],
          [0.1057, 0.9907, 0.5937, 0.5606, 0.2390, 0.4718, 0.4294, 0.8508],
          [0.1057, 0.9907, 0.5937, 0.5606, 0.2390, 0.4718, 0.4294, 0.8508],
          [0.1057, 0.9907, 0.5937, 0.5606, 0.2390, 0.4718, 0.4294, 0.8508]],

         [[0.1053, 0.9353, 0.5285, 0.5943, 0.8888, 0.2838, 0.6447, 0.6535],
          [0.1053, 0.9353, 0.5285, 0.5943, 0.8888, 0.2838, 0.6447, 0.6535],
          [0.1053, 0.9353, 0.5285, 0.5943, 0.8888, 0.2838, 0.6447, 0.6535],
          [0.1053, 0.9353, 0.5285, 0.5943, 0.8888, 0.2838, 0.6447, 0.6535]],

         [[0.6196, 0.0497, 0.5953, 0.9974, 0.6720, 0.4915, 0.6581, 0.6456],
      